# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, process, and visualize a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset metadata is accessible as a Croissant JSON-LD schema from the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print summary details
print(f"Dataset Title: {metadata.name}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}\n")
print(f"Authors: {getattr(metadata, 'author', None)}")
print(f"License: {metadata.license}")

## 2. Data Overview
Explore the structure of the dataset: view the available record sets, fields, and their corresponding `@id` references as defined in the Croissant schema.

Below, all entities are referenced strictly by their `@id`.

In [ ]:
# List all record sets and their field/column structure by @id
record_sets = dataset.record_sets()
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"\nRecord set @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        # List fields by @id
        print("  Fields/Columns:")
        for f in rs.get('fields', []):
            print(f"   - Field @id: {f['@id']} | Label: {f.get('name', f.get('label', ''))}")
else:
    print('No record sets present in metadata.')

## 3. Data Extraction
We now load data for each record set defined in the Croissant package, referencing each exclusively by its `@id`.

Below, we create a dictionary of DataFrames keyed by record set `@id`. Replace `<record_set_id>` with the specific `@id` you wish to inspect.

In [ ]:
# Collect all available record set @ids
record_sets = dataset.record_sets()

record_set_ids = [rs['@id'] for rs in record_sets]
print("Record set @ids found:")
for rid in record_set_ids:
    print(f"- {rid}")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {type(e).__name__} - {e}")

if dataframes:
    # Print column list for the first available record set
    first_rs = record_set_ids[0]
    print(f"\nColumns in first record set ({first_rs}):")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No dataframes could be loaded. Please check schema.")

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA techniques such as filtering by field values, normalization, and grouping, using the `@id` of each field.

- Filter a numeric field (referenced by its `@id`)
- Normalize it
- Show grouping/aggregation (by another field's `@id`, if present)

In [ ]:
# Choose a record set to explore (replace with actual @id from previous step)
example_rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(example_rs_id)

if df is not None and not df.empty:
    print(f"Available columns in {example_rs_id}:")
    print(list(df.columns))
    # Try to infer a numeric field by @id (here: choose the first that looks numeric)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No obvious numeric field found; cannot perform EDA.")
    else:
        # Filter values above threshold
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        colnorm = f"{numeric_field_id}_normalized"
        filtered_df[colnorm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nFirst 5 entries of normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, colnorm]].head())

        # Group by another field if available (e.g., first string/categorical field by @id)
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field for aggregation available.")
else:
    print("No data available for EDA.")

## 5. Visualization
Below, we plot a histogram of the numeric field, and (if grouping is possible) a bar chart by group, referencing all columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Grouped bar plot if group field is available
    if group_field_id and group_field_id in df.columns:
        means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10,5))
        sns.barplot(data=means, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading and exploring a multi-record-set Croissant dataset using the `mlcroissant` library, referencing all entities by their `@id`.

- Dataset fields, record sets, and their columns were discovered and loaded.
- Key numeric fields were processed, normalized, and visualized.
- All data manipulations were referenced using the Croissant schema's unique `@id` system.

You may now apply similar methods for further analysis, modeling, or integration with downstream applications.